# 14 — Factor Decomposition of the 2021-22 Reversal

**Goal:** Show that the sentiment long-short spread's losses in 2021 and 2022 reflect
a rotation in style factors (value, profitability, investment), not a sentiment-specific
alpha. This is a diagnostic that backs the Discussion's reading of the regime break. It is
read-only on the existing return series and changes no headline result.

**Inputs:** `data/long_short_returns.parquet`, `data/factors.parquet`

**Outputs:**
- `output/table_factor_rotation_2021_22.csv` (per-window summary)
- `output/table_factor_contributions_2021_22.csv` (per-factor contribution, 2021-22)

**Method:** estimate the full-window (2013-2023) six-factor loadings, then for each
sub-window decompose the realised long-short return into the part predicted by the
full-window loadings times the realised factor returns, plus a residual (the sub-window
alpha). A large factor part and a small, insignificant residual mean the losses are
style-driven rather than sentiment-specific.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA = Path.home() / "thesis" / "data"
OUTPUT = Path.home() / "thesis" / "output"
FACTORS = ["mkt_rf", "smb", "hml", "rmw", "cma", "mom"]

ls = pd.read_parquet(DATA / "long_short_returns.parquet")
ls["date"] = pd.to_datetime(ls["date"])
factors = pd.read_parquet(DATA / "factors.parquet")
reg = ls.merge(factors[["date"] + FACTORS], on="date", how="inner")

def fit_6factor(d):
    """OLS of the long-short on the six factors, Newey-West (3 lags)."""
    y = d["long_short"].astype("float64")
    X = sm.add_constant(d[FACTORS].astype("float64"))
    return sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})

# Full window 2013-2023 -- these loadings are the chapter's Table 3 betas
full = reg[(reg["date"].dt.year >= 2013) & (reg["date"].dt.year <= 2023)]
full_res = fit_6factor(full)
print(f"Verify kernel: {__import__('sys').executable}")
print(f"Full window 2013-2023: {int(full_res.nobs)} months, R2={full_res.rsquared:.2f}")
for k in ["const"] + FACTORS:
    print(f"  {k:7s} beta={full_res.params[k]:+.3f}  t={full_res.tvalues[k]:+.2f}")

Verify kernel: /opt/anaconda3/envs/thesis/bin/python
Full window 2013-2023: 132 months, R2=0.35
  const   beta=+0.001  t=+1.37
  mkt_rf  beta=+0.054  t=+1.57
  smb     beta=-0.225  t=-3.59
  hml     beta=-0.092  t=-1.88
  rmw     beta=-0.164  t=-2.11
  cma     beta=-0.196  t=-2.59
  mom     beta=-0.079  t=-1.74


In [2]:
# Decompose each sub-window: realised LS vs the part predicted by full-window betas.
def decompose(years, label):
    d = reg[reg["date"].dt.year.isin(years)]
    realised = d["long_short"].mean()
    contrib = {f: full_res.params[f] * d[f].mean() for f in FACTORS}   # beta_full * realised factor
    factor_part = sum(contrib.values())
    deviation = realised - full_res.params["const"]                    # realised minus long-run alpha
    pct_explained = factor_part / deviation * 100 if abs(deviation) > 1e-9 else np.nan
    sub = fit_6factor(d) if len(d) >= 12 else None                     # re-estimated window alpha (noisy when short)
    row = {
        "window": label, "n_months": len(d),
        "mean_ls_pct_mo": realised * 100,
        "factor_part_pct_mo": factor_part * 100,
        "pct_of_shortfall_explained": pct_explained,
        "subwindow_alpha_pct_yr": sub.params["const"] * 12 * 100 if sub is not None else np.nan,
        "subwindow_alpha_t": sub.tvalues["const"] if sub is not None else np.nan,
    }
    return row, contrib

windows = [("2013-2023", range(2013, 2024)), ("2021-2022", [2021, 2022]),
           ("2021", [2021]), ("2022", [2022])]
rows, contribs_2122 = [], None
for label, yrs in windows:
    row, contrib = decompose(list(yrs), label)
    rows.append(row)
    if label == "2021-2022":
        contribs_2122 = contrib
    print(f"{label:10s} mean LS {row['mean_ls_pct_mo']:+.3f}%/mo | factor part {row['factor_part_pct_mo']:+.3f} | "
          f"explains {row['pct_of_shortfall_explained']:.0f}% | window alpha {row['subwindow_alpha_pct_yr']:+.1f}%/yr (t={row['subwindow_alpha_t']:+.2f})")

2013-2023  mean LS +0.150%/mo | factor part +0.031 | explains 100% | window alpha +1.4%/yr (t=+1.37)
2021-2022  mean LS -0.404%/mo | factor part -0.742 | explains 142% | window alpha +3.0%/yr (t=+0.84)
2021       mean LS -0.157%/mo | factor part -0.505 | explains 183% | window alpha +16.5%/yr (t=+8.38)
2022       mean LS -0.650%/mo | factor part -0.979 | explains 127% | window alpha +3.5%/yr (t=+0.92)


In [3]:
# Save the per-window summary and the 2021-22 per-factor contribution breakdown.
summary = pd.DataFrame(rows).round(3)
summary.to_csv(OUTPUT / "table_factor_rotation_2021_22.csv", index=False)
print(summary.to_string(index=False))

contrib_df = pd.DataFrame({
    "factor": FACTORS,
    "full_beta": [full_res.params[f] for f in FACTORS],
    "realised_factor_pct_mo": [reg[reg["date"].dt.year.isin([2021, 2022])][f].mean() * 100 for f in FACTORS],
    "contribution_pct_mo": [contribs_2122[f] * 100 for f in FACTORS],
}).round(3).sort_values("contribution_pct_mo")
contrib_df.to_csv(OUTPUT / "table_factor_contributions_2021_22.csv", index=False)
print("\n2021-22 per-factor contribution to the long-short (most negative first):")
print(contrib_df.to_string(index=False))
print(f"\nValue/quality factors (HML, RMW, CMA) drive the loss; SMB (size) contributes "
      f"{contribs_2122['smb']*100:+.3f}%/mo, essentially nil.")

   window  n_months  mean_ls_pct_mo  factor_part_pct_mo  pct_of_shortfall_explained  subwindow_alpha_pct_yr  subwindow_alpha_t
2013-2023       132           0.150               0.031                     100.000                   1.425              1.373
2021-2022        24          -0.404              -0.742                     141.995                   2.992              0.842
     2021        12          -0.157              -0.505                     182.899                  16.514              8.376
     2022        12          -0.650              -0.979                     127.315                   3.482              0.925

2021-22 per-factor contribution to the long-short (most negative first):
factor  full_beta  realised_factor_pct_mo  contribution_pct_mo
   cma     -0.196                   1.534               -0.301
   rmw     -0.164                   1.305               -0.214
   hml     -0.092                   2.120               -0.194
   mom     -0.079                   0.7